# BERT IELTS Multi-Trait Scoring — Essay Only, 4 Trait Stacks

In [1]:
# =========================
# INSTALL + IMPORTS
# =========================

!pip -q install transformers

import os
import re
import math
import random
from dataclasses import dataclass
from typing import List, Dict, Tuple
from collections import Counter

import numpy as np
import pandas as pd

from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import cohen_kappa_score, mean_absolute_error

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup

try:
    from IPython.display import display
except Exception:
    display = print


In [2]:
# =========================
# REPRODUCIBILITY
# =========================

SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(SEED)


In [3]:
# =========================
# CONFIG - BERT ESSAY-ONLY, 4 TRAIT STACKS
# =========================

OUTPUT_DIR = "/content/bert_essay_only_trait_stacks"
os.makedirs(OUTPUT_DIR, exist_ok=True)

TRAIN_PATH = "/content/train.csv"
VAL_PATH   = "/content/val.csv"
TEST_PATH  = "/content/test.csv"

TRAIT_COLS = ["TR", "CC", "LR", "GRA"]

MIN_BAND = 0.0
MAX_BAND = 9.0

MODEL_NAME = "bert-base-uncased"
MAX_LEN = 512

STACK_HIDDEN = 512
DROPOUT = 0.2

BATCH_SIZE = 48
EPOCHS = 20
LR = 2e-5
WEIGHT_DECAY = 0.01
MAX_GRAD_NORM = 1.0
WARMUP_RATIO = 0.1

EARLY_STOPPING_PATIENCE = 4
MIN_DELTA = 1e-4

USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS = False
OVERALL_CONSISTENCY_WEIGHT = 0.0

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)


DEVICE: cuda


In [4]:
# =========================
# LOAD PRE-SPLIT CSV FILES
# Upload train.csv, val.csv, test.csv to /content first
# =========================

train_df = pd.read_csv(TRAIN_PATH)
val_df   = pd.read_csv(VAL_PATH)
test_df  = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Val:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain columns:")
print(train_df.columns.tolist())

display(train_df.head())

# =========================
# BASIC COLUMN CHECK
# =========================

required_cols = ["essay", "TR", "CC", "LR", "GRA"]

for name, df_ in [
    ("train_df", train_df),
    ("val_df", val_df),
    ("test_df", test_df),
]:
    missing = [c for c in required_cols if c not in df_.columns]

    if missing:
        raise ValueError(f"{name} is missing columns: {missing}")

    df_["essay"] = df_["essay"].fillna("").astype(str)

    for col in TRAIT_COLS:
        df_[col] = df_[col].astype(float)

    if "Overall" in df_.columns:
        df_["Overall"] = df_["Overall"].astype(float)

print("All datasets are ready.")

# =========================
# OVERLAP CHECK
# =========================

train_essays = set(train_df["essay"].astype(str))
val_essays = set(val_df["essay"].astype(str))
test_essays = set(test_df["essay"].astype(str))

print("Train-Val overlap:", len(train_essays & val_essays))
print("Train-Test overlap:", len(train_essays & test_essays))
print("Val-Test overlap:", len(val_essays & test_essays))


Train: (7743, 17)
Val: (969, 17)
Test: (970, 17)

Train columns:
['prompt', 'essay', 'evaluation', 'essay_len', 'TR', 'CC', 'LR', 'GRA', 'n_found', 'parse_quality', 'overall_raw', 'band', 'prompt_relevance', 'lexical_diversity', 'readability_score', 'target_text', 'source']


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,overall_raw,band,prompt_relevance,lexical_diversity,readability_score,target_text,source
0,Some employers believe that job applicants’ so...,There has been much discussion revolving aroun...,Task Achievement:\r\n\r\n- The candidate has e...,273,7.5,7.0,7.0,7.0,4,good,7.125,7.0,0.592603,0.581818,40.754804,Analysis: This essay has a lexical diversity o...,hf
1,Some people believe that teenagers should be r...,A nation is known as a vast garden and childre...,Task Achievement:\r\nThe essay effectively add...,324,4.0,4.0,3.0,3.0,4,good,3.500,3.5,0.600165,0.563077,35.677525,Analysis: This essay has a lexical diversity o...,hf
2,some people say that economic growth is the on...,"In the modern world ,a burning issue arises th...",Task Achievement:\r\n\r\nThe essay adequately ...,348,5.5,5.0,5.0,5.0,4,good,5.125,5.0,0.746589,0.452450,40.414844,Analysis: This essay has a lexical diversity o...,hf
3,Some believe that people should make efforts t...,The issue of climate change was always debatab...,Task Achievement:\r\n- The candidate has adequ...,307,7.0,7.0,6.0,6.0,4,good,6.500,6.5,0.628859,0.629870,39.378580,Analysis: This essay has a lexical diversity o...,hf
4,Nowadays families move to different countries ...,Immigrating to other nations for business purp...,Task Achievement:\r\nThe candidate has effecti...,311,7.5,8.0,7.0,7.5,4,good,7.500,7.5,0.586910,0.612179,40.385892,Analysis: This essay has a lexical diversity o...,hf


All datasets are ready.
Train-Val overlap: 0
Train-Test overlap: 1
Val-Test overlap: 0


In [5]:
# =========================
# METRIC FUNCTIONS
# Includes derived Overall QWK
# =========================

def band_to_scaled(x):
    x = np.asarray(x, dtype=np.float32)
    return (x - MIN_BAND) / (MAX_BAND - MIN_BAND)


def scaled_to_band(x):
    x = np.asarray(x, dtype=np.float32)

    band = x * (MAX_BAND - MIN_BAND) + MIN_BAND
    band = np.clip(band, MIN_BAND, MAX_BAND)

    # IELTS half-band rounding
    band = np.round(band * 2) / 2

    return band


def qwk_band(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=np.float32)
    y_pred = np.asarray(y_pred, dtype=np.float32)

    valid = ~np.isnan(y_true) & ~np.isnan(y_pred)

    y_true = y_true[valid]
    y_pred = y_pred[valid]

    if len(y_true) == 0:
        return np.nan

    y_true_int = (y_true * 2).astype(int)
    y_pred_int = (y_pred * 2).astype(int)

    return cohen_kappa_score(
        y_true_int,
        y_pred_int,
        weights="quadratic",
    )


def compute_metrics(y_true_scaled, y_pred_scaled, overall_true=None):
    y_true_band = scaled_to_band(y_true_scaled)
    y_pred_band = scaled_to_band(y_pred_scaled)

    metrics = {}

    trait_qwks = []
    trait_maes = []

    for i, trait in enumerate(TRAIT_COLS):
        qwk = qwk_band(y_true_band[:, i], y_pred_band[:, i])
        mae = mean_absolute_error(y_true_band[:, i], y_pred_band[:, i])

        metrics[f"{trait}_qwk"] = qwk
        metrics[f"{trait}_mae"] = mae

        trait_qwks.append(qwk)
        trait_maes.append(mae)

    metrics["mean_trait_qwk"] = float(np.nanmean(trait_qwks))
    metrics["mean_trait_mae"] = float(np.nanmean(trait_maes))

    # Derived overall = average of 4 predicted trait bands
    derived_overall_pred = np.mean(y_pred_band, axis=1)
    derived_overall_pred = np.round(derived_overall_pred * 2) / 2
    derived_overall_pred = np.clip(derived_overall_pred, MIN_BAND, MAX_BAND)

    # Derived overall compared to derived gold overall from gold traits
    derived_overall_gold_from_traits = np.mean(y_true_band, axis=1)
    derived_overall_gold_from_traits = np.round(derived_overall_gold_from_traits * 2) / 2
    derived_overall_gold_from_traits = np.clip(derived_overall_gold_from_traits, MIN_BAND, MAX_BAND)

    metrics["derived_overall_from_traits_qwk"] = qwk_band(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    metrics["derived_overall_from_traits_mae"] = mean_absolute_error(
        derived_overall_gold_from_traits,
        derived_overall_pred,
    )

    # Derived overall compared to real Overall column, if available
    if overall_true is not None:
        overall_true = np.asarray(overall_true, dtype=np.float32)
        valid = ~np.isnan(overall_true)

        if valid.any():
            metrics["derived_overall_qwk"] = qwk_band(
                overall_true[valid],
                derived_overall_pred[valid],
            )

            metrics["derived_overall_mae"] = mean_absolute_error(
                overall_true[valid],
                derived_overall_pred[valid],
            )

    return metrics, y_true_band, y_pred_band, derived_overall_pred


In [6]:
# =========================
# TOKENIZER
# =========================

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print("Loaded tokenizer:", MODEL_NAME)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loaded tokenizer: bert-base-uncased


In [7]:
# =========================
# DATASET + COLLATOR - ESSAY ONLY
# =========================

class IELTSBertEssayOnlyDataset(Dataset):
    def __init__(self, df, tokenizer, max_len=512):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]

        encoded = self.tokenizer(
            str(row["essay"]),
            truncation=True,
            padding=False,
            max_length=self.max_len,
        )

        labels_band = row[TRAIT_COLS].astype(float).values.astype(np.float32)
        labels_scaled = band_to_scaled(labels_band).astype(np.float32)

        if "Overall" in self.df.columns and not pd.isna(row.get("Overall", np.nan)):
            overall_band = np.float32(row["Overall"])
        else:
            overall_band = np.float32(np.nan)

        return {
            "input_ids": encoded["input_ids"],
            "attention_mask": encoded["attention_mask"],
            "labels": labels_scaled,
            "overall_band": overall_band,
        }


@dataclass
class BertDataCollator:
    tokenizer: object

    def __call__(self, batch):
        features = [
            {
                "input_ids": x["input_ids"],
                "attention_mask": x["attention_mask"],
            }
            for x in batch
        ]

        padded = self.tokenizer.pad(
            features,
            padding=True,
            return_tensors="pt",
        )

        padded["labels"] = torch.tensor(
            np.stack([x["labels"] for x in batch]),
            dtype=torch.float32,
        )

        padded["overall_band"] = torch.tensor(
            [x["overall_band"] for x in batch],
            dtype=torch.float32,
        )

        return padded


train_dataset = IELTSBertEssayOnlyDataset(train_df, tokenizer, max_len=MAX_LEN)
val_dataset   = IELTSBertEssayOnlyDataset(val_df, tokenizer, max_len=MAX_LEN)
test_dataset  = IELTSBertEssayOnlyDataset(test_df, tokenizer, max_len=MAX_LEN)

collator = BertDataCollator(tokenizer=tokenizer)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collator)
val_loader   = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)
test_loader  = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collator)


In [8]:
# =========================
# MODEL - BERT ESSAY ONLY + 4 TRAIT STACKS
# =========================

class TraitStack(nn.Module):
    def __init__(self, input_dim, hidden_dim=512, dropout=0.2):
        super().__init__()

        self.net = nn.Sequential(
            nn.LayerNorm(input_dim),

            nn.Linear(input_dim, hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim, hidden_dim // 2),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 2, hidden_dim // 4),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(hidden_dim // 4, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


class BertEssayOnlyTraitScorer(nn.Module):
    def __init__(self, model_name, stack_hidden=512, dropout=0.2):
        super().__init__()

        self.bert = AutoModel.from_pretrained(model_name)
        bert_hidden = self.bert.config.hidden_size

        self.dropout = nn.Dropout(dropout)
        self.norm = nn.LayerNorm(bert_hidden)

        self.stacks = nn.ModuleDict({
            trait: TraitStack(
                input_dim=bert_hidden,
                hidden_dim=stack_hidden,
                dropout=dropout,
            )
            for trait in TRAIT_COLS
        })

    def mean_pooling(self, last_hidden_state, attention_mask):
        mask = attention_mask.unsqueeze(-1).float()
        summed = (last_hidden_state * mask).sum(dim=1)
        counts = mask.sum(dim=1).clamp(min=1.0)
        return summed / counts

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )

        # Mean pooling is usually stable for essay scoring
        essay_repr = self.mean_pooling(outputs.last_hidden_state, attention_mask)
        essay_repr = self.norm(essay_repr)
        essay_repr = self.dropout(essay_repr)

        preds = []
        for trait in TRAIT_COLS:
            preds.append(self.stacks[trait](essay_repr))

        preds = torch.stack(preds, dim=1)
        preds = torch.sigmoid(preds)

        return preds


model = BertEssayOnlyTraitScorer(
    model_name=MODEL_NAME,
    stack_hidden=STACK_HIDDEN,
    dropout=DROPOUT,
).to(DEVICE)

print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total params: {total_params:,}")
print(f"Trainable params: {trainable_params:,}")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


BertEssayOnlyTraitScorer(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12, elem

In [9]:
# =========================
# LOSS
# =========================

def criterion_loss(pred_scaled, labels_scaled, overall_band=None):
    loss = 0.0

    for i, trait in enumerate(TRAIT_COLS):
        loss = loss + F.mse_loss(pred_scaled[:, i], labels_scaled[:, i])

    loss = loss / len(TRAIT_COLS)

    if USE_OPTIONAL_OVERALL_CONSISTENCY_LOSS and overall_band is not None:
        valid = ~torch.isnan(overall_band)

        if valid.any():
            pred_overall_scaled = pred_scaled.mean(dim=1)

            gold_overall_scaled = (
                overall_band.to(pred_scaled.device) - MIN_BAND
            ) / (MAX_BAND - MIN_BAND)

            loss = loss + OVERALL_CONSISTENCY_WEIGHT * F.mse_loss(
                pred_overall_scaled[valid],
                gold_overall_scaled[valid],
            )

    return loss


In [10]:
# =========================
# EVALUATION HELPERS - ESSAY ONLY
# =========================

def move_batch_to_device(batch):
    out = {}

    for k, v in batch.items():
        if isinstance(v, torch.Tensor):
            out[k] = v.to(DEVICE)
        else:
            out[k] = v

    return out


@torch.no_grad()
def predict_loader(model, loader):
    model.eval()

    all_preds = []
    all_labels = []
    all_overall = []

    for batch in tqdm(loader, desc="Predict"):
        batch = move_batch_to_device(batch)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        all_preds.append(preds.detach().cpu().numpy())
        all_labels.append(batch["labels"].detach().cpu().numpy())
        all_overall.append(batch["overall_band"].detach().cpu().numpy())

    return (
        np.concatenate(all_labels, axis=0),
        np.concatenate(all_preds, axis=0),
        np.concatenate(all_overall, axis=0),
    )


def print_metrics(metrics, title):
    print("\n" + title)
    print("=" * len(title))

    for k, v in metrics.items():
        if isinstance(v, float):
            print(f"{k}: {v:.4f}")
        else:
            print(f"{k}: {v}")


def evaluate_model(model, loader, name="val"):
    y_true, y_pred, overall_true = predict_loader(model, loader)

    metrics, y_true_band, y_pred_band, overall_pred = compute_metrics(
        y_true,
        y_pred,
        overall_true,
    )

    print_metrics(metrics, f"== {name} ==")

    return metrics, y_true_band, y_pred_band, overall_pred


In [11]:
# =========================
# TRAIN BERT ESSAY-ONLY MODEL WITH EARLY STOPPING
# Main metric: mean_trait_qwk
# =========================

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY,
)

total_training_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_training_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_training_steps,
)

epochs_no_improve = 0
best_val_qwk = -1.0

best_path = os.path.join(OUTPUT_DIR, "best_bert_essay_only.pt")

for epoch in range(1, EPOCHS + 1):
    model.train()

    running_loss = 0.0

    pbar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")

    for step, batch in enumerate(pbar, start=1):
        batch = move_batch_to_device(batch)

        optimizer.zero_grad(set_to_none=True)

        preds = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
        )

        loss = criterion_loss(
            preds,
            batch["labels"],
            batch["overall_band"],
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            MAX_GRAD_NORM,
        )

        optimizer.step()
        scheduler.step()

        running_loss += loss.item()
        avg_running_loss = running_loss / step

        pbar.set_postfix({
            "loss": f"{avg_running_loss:.6f}",
            "lr": optimizer.param_groups[0]["lr"],
        })

    avg_train_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch} train loss: {avg_train_loss:.6f}")

    val_metrics, _, _, _ = evaluate_model(
        model,
        val_loader,
        name=f"val epoch {epoch}",
    )

    current_qwk = val_metrics["mean_trait_qwk"]
    selected_metric_name = "mean_trait_qwk"

    print(f"Selected validation metric: {selected_metric_name}")
    print(f"Current val QWK: {current_qwk:.4f}")
    print(f"Best val QWK so far: {best_val_qwk:.4f}")

    if "derived_overall_qwk" in val_metrics:
        print(f"Val derived_overall_qwk: {val_metrics['derived_overall_qwk']:.4f}")

    improved = current_qwk > best_val_qwk + MIN_DELTA

    if improved:
        best_val_qwk = current_qwk
        epochs_no_improve = 0

        torch.save(
            {
                "epoch": epoch,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "best_val_qwk": best_val_qwk,
                "selected_metric_name": selected_metric_name,
                "best_val_metrics": val_metrics,
                "config": {
                    "FEATURE_MODE": "bert_essay_only",
                    "MODEL_NAME": MODEL_NAME,
                    "MAX_LEN": MAX_LEN,
                    "STACK_HIDDEN": STACK_HIDDEN,
                    "DROPOUT": DROPOUT,
                    "BATCH_SIZE": BATCH_SIZE,
                    "EPOCHS": EPOCHS,
                    "LR": LR,
                    "WEIGHT_DECAY": WEIGHT_DECAY,
                    "MAX_GRAD_NORM": MAX_GRAD_NORM,
                    "TRAIT_COLS": TRAIT_COLS,
                    "MIN_BAND": MIN_BAND,
                    "MAX_BAND": MAX_BAND,
                    "EARLY_STOPPING_PATIENCE": EARLY_STOPPING_PATIENCE,
                    "MIN_DELTA": MIN_DELTA,
                },
            },
            best_path,
        )

        print(f"[SAVE] Best model saved to: {best_path}")
        print(f"[SAVE] best_val_qwk = {best_val_qwk:.4f}")

    else:
        epochs_no_improve += 1

        print(
            f"[EARLY STOPPING] No improvement for "
            f"{epochs_no_improve}/{EARLY_STOPPING_PATIENCE} epochs."
        )

        if epochs_no_improve >= EARLY_STOPPING_PATIENCE:
            print("[EARLY STOPPING] Triggered.")
            print(f"Stopped at epoch {epoch}.")
            break

    print("-" * 80)

print("\nTraining finished.")
print(f"Best validation QWK: {best_val_qwk:.4f}")
print(f"Best checkpoint path: {best_path}")


Epoch 1/20:   0%|          | 0/162 [00:00<?, ?it/s]


Epoch 1 train loss: 0.042429


Predict:   0%|          | 0/21 [00:00<?, ?it/s]


== val epoch 1 ==
TR_qwk: 0.1668
TR_mae: 1.0604
CC_qwk: 0.2004
CC_mae: 1.1796
LR_qwk: 0.2635
LR_mae: 1.0784
GRA_qwk: 0.2687
GRA_mae: 1.1486
mean_trait_qwk: 0.2249
mean_trait_mae: 1.1167
derived_overall_from_traits_qwk: 0.2351
derived_overall_from_traits_mae: 1.0960
Selected validation metric: mean_trait_qwk
Current val QWK: 0.2249
Best val QWK so far: -1.0000
[SAVE] Best model saved to: /content/bert_essay_only_trait_stacks/best_bert_essay_only.pt
[SAVE] best_val_qwk = 0.2249
--------------------------------------------------------------------------------


Epoch 2/20:   0%|          | 0/162 [00:00<?, ?it/s]

...


== val epoch 9 ==
TR_qwk: 0.5382
TR_mae: 1.0516
CC_qwk: 0.5499
CC_mae: 1.1496
LR_qwk: 0.5806
LR_mae: 1.0624
GRA_qwk: 0.5892
GRA_mae: 1.1027
mean_trait_qwk: 0.5645
mean_trait_mae: 1.0916
derived_overall_from_traits_qwk: 0.5638
derived_overall_from_traits_mae: 1.0738
Selected validation metric: mean_trait_qwk
Current val QWK: 0.5645
Best val QWK so far: 0.5964
[EARLY STOPPING] No improvement for 3/4 epochs.
--------------------------------------------------------------------------------


Epoch 10/20:   0%|          | 0/162 [00:00<?, ?it/s]


Epoch 10 train loss: 0.004615


Predict:   0%|          | 0/21 [00:00<?, ?it/s]


== val epoch 10 ==
TR_qwk: 0.5509
TR_mae: 0.9974
CC_qwk: 0.5621
CC_mae: 1.1084
LR_qwk: 0.5896
LR_mae: 0.9954
GRA_qwk: 0.5971
GRA_mae: 1.0366
mean_trait_qwk: 0.5750
mean_trait_mae: 1.0344
derived_overall_from_traits_qwk: 0.5816
derived_overall_from_traits_mae: 1.0077
Selected validation metric: mean_trait_qwk
Current val QWK: 0.5750
Best val QWK so far: 0.5964
[EARLY STOPPING] No improvement for 4/4 epochs.
[EARLY STOPPING] Triggered.
Stopped at epoch 10.

Training finished.
Best validation QWK: 0.5964
Best checkpoint path: /content/bert_essay_only_trait_stacks/best_bert_essay_only.pt


In [12]:
# =========================
# LOAD BEST CHECKPOINT AND EVALUATE ON TEST SET
# =========================

checkpoint = torch.load(
    best_path,
    map_location=DEVICE,
    weights_only=False,
)

model.load_state_dict(checkpoint["model_state_dict"])
model.to(DEVICE)

print("Loaded checkpoint from:", best_path)
print("Best epoch:", checkpoint.get("epoch"))
print("Best validation QWK:", checkpoint.get("best_val_qwk"))
print("Selected metric:", checkpoint.get("selected_metric_name"))

test_metrics, y_true_band, y_pred_band, overall_pred = evaluate_model(
    model,
    test_loader,
    name="test best BERT essay-only",
)

# =========================
# EXPORT TEST PREDICTIONS
# =========================

def build_prediction_df(df, y_true_band, y_pred_band, overall_pred):
    out = df.reset_index(drop=True).copy()

    for i, trait in enumerate(TRAIT_COLS):
        out[f"gold_{trait}"] = y_true_band[:, i]
        out[f"pred_{trait}"] = y_pred_band[:, i]
        out[f"abs_error_{trait}"] = (
            out[f"pred_{trait}"] - out[f"gold_{trait}"]
        ).abs()

    out["pred_Overall_derived"] = overall_pred

    if "Overall" in out.columns:
        out["gold_Overall"] = out["Overall"]

        valid = ~pd.isna(out["gold_Overall"])

        if valid.any():
            print(
                "Derived Overall QWK:",
                qwk_band(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

            print(
                "Derived Overall MAE:",
                mean_absolute_error(
                    out.loc[valid, "gold_Overall"],
                    out.loc[valid, "pred_Overall_derived"],
                ),
            )

    out["mean_abs_trait_error"] = out[
        [f"abs_error_{t}" for t in TRAIT_COLS]
    ].mean(axis=1)

    return out


pred_df = build_prediction_df(
    test_df,
    y_true_band,
    y_pred_band,
    overall_pred,
)

pred_path = os.path.join(
    OUTPUT_DIR,
    "test_predictions_bert_essay_only_trait_stacks.csv",
)

pred_df.to_csv(pred_path, index=False)

print("Saved:", pred_path)
display(pred_df.head())

# =========================
# EXPORT METRIC SUMMARY
# =========================

summary_rows = []

for trait in TRAIT_COLS:
    qwk = qwk_band(
        pred_df[f"gold_{trait}"],
        pred_df[f"pred_{trait}"],
    )

    mae = mean_absolute_error(
        pred_df[f"gold_{trait}"],
        pred_df[f"pred_{trait}"],
    )

    summary_rows.append({
        "model": "BERT essay-only",
        "target": trait,
        "QWK": qwk,
        "MAE": mae,
    })

if "gold_Overall" in pred_df.columns:
    valid = ~pd.isna(pred_df["gold_Overall"])

    if valid.any():
        qwk = qwk_band(
            pred_df.loc[valid, "gold_Overall"],
            pred_df.loc[valid, "pred_Overall_derived"],
        )

        mae = mean_absolute_error(
            pred_df.loc[valid, "gold_Overall"],
            pred_df.loc[valid, "pred_Overall_derived"],
        )

        summary_rows.append({
            "model": "BERT essay-only",
            "target": "Overall_derived",
            "QWK": qwk,
            "MAE": mae,
        })

summary_df = pd.DataFrame(summary_rows)

summary_path = os.path.join(
    OUTPUT_DIR,
    "test_metric_summary_bert_essay_only.csv",
)

summary_df.to_csv(summary_path, index=False)

print("Saved:", summary_path)
display(summary_df)


Loaded checkpoint from: /content/bert_essay_only_trait_stacks/best_bert_essay_only.pt
Best epoch: 6
Best validation QWK: 0.5963751136370758
Selected metric: mean_trait_qwk


Predict:   0%|          | 0/21 [00:00<?, ?it/s]


== test best BERT essay-only ==
TR_qwk: 0.5292
TR_mae: 1.1371
CC_qwk: 0.5401
CC_mae: 1.2433
LR_qwk: 0.5666
LR_mae: 1.1546
GRA_qwk: 0.5818
GRA_mae: 1.1990
mean_trait_qwk: 0.5544
mean_trait_mae: 1.1835
derived_overall_from_traits_qwk: 0.5592
derived_overall_from_traits_mae: 1.1680
Saved: /content/bert_essay_only_trait_stacks/test_predictions_bert_essay_only_trait_stacks.csv


,prompt,essay,evaluation,essay_len,TR,CC,LR,GRA,n_found,parse_quality,...,pred_CC,abs_error_CC,gold_LR,pred_LR,abs_error_LR,gold_GRA,pred_GRA,abs_error_GRA,pred_Overall_derived,mean_abs_trait_error
0,18.Education of young people is highly priorit...,It can be argued that the vast majority of cou...,Task Achievement: \r\nThe essay adequately add...,303,7.0,7.0,6.0,6.0,4,good,...,8.0,1.0,6.0,7.5,1.5,6.0,8.0,2.0,8.0,1.375
1,Nowadays in many countries most shops and prod...,"In today's modern world, daily use things are ...",Task Achievement:\r\nThe candidate has adequat...,300,6.0,5.5,5.5,5.5,4,good,...,3.5,2.0,5.5,3.5,2.0,5.5,3.0,2.5,3.5,2.125
2,Some people believe that the rage of technolog...,Technology is deeply relative to the living of...,Task Achievement: 5.5\r\n- The candidate has e...,296,5.5,6.0,6.0,6.0,4,good,...,8.0,2.0,6.0,7.5,1.5,6.0,7.5,1.5,8.0,1.875
3,The world has many towns and cites constructed...,Numerous cities have houses and replanning and...,Task Achievement:\r\n\r\nThe essay adequately ...,342,6.5,6.5,6.0,6.0,4,good,...,5.0,1.5,6.0,5.0,1.0,6.0,5.0,1.0,5.0,1.125
4,Many people argue that in order to improve the...,"In this modern world, the importance of higher...",Task Achievement: (Band Score: 4)\r\n- The can...,264,4.0,3.5,3.0,3.0,4,good,...,6.5,3.0,3.0,6.0,3.0,3.0,6.0,3.0,6.0,2.875


Saved: /content/bert_essay_only_trait_stacks/test_metric_summary_bert_essay_only.csv


,model,target,QWK,MAE
0,BERT essay-only,TR,0.529198,1.137113
1,BERT essay-only,CC,0.540094,1.243299
2,BERT essay-only,LR,0.566623,1.154639
3,BERT essay-only,GRA,0.581834,1.198969
